# 🚀 Semana 10 · Unidad 3 — Quicksort

## Información del Curso

| Aspecto | Detalle |
|--------|--------|
| **Universidad** | Universidad de Talca, Chile |
| **Carrera** | Ingeniería Civil en Informática |
| **Semestre** | 2°-3° año |
| **Curso** | Algoritmos y Estructuras de Datos |
| **Docente** | PhD. César Astudillo |
| **Clase** | Semana 10 · Unidad 3 — Quicksort |
| **Duración** | 50 minutos |

---
> 🎯 *Este notebook está diseñado para ser ejecutado en clase de forma interactiva.  
> Ejecuta las celdas en orden de arriba hacia abajo.*

## Verificación de Dependencias

In [ ]:
import random
import time
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches ## Para crear leyendas personalizadas
import numpy as np
from IPython.display import display, HTML ## Para mostrar tablas de manera más atractiva
print("✅ Dependencias cargadas correctamente")

## 🎯 Objetivos de Aprendizaje

Al finalizar esta sesión, el estudiante será capaz de:

1. **Explicar** el concepto de partición como núcleo de Quicksort.
2. **Implementar** la partición de dos punteros y Quicksort completo sobre ella.
3. **Analizar** el impacto del pivote en la complejidad: O(n log n) promedio vs O(n²) peor caso.
4. **Comparar** distintas estrategias de selección de pivote (primero, aleatorio, mediana de tres).
5. **Aplicar** la partición tricotómica cuando hay muchas claves repetidas.
6. **Contrastar** Quicksort con Merge Sort en complejidad, memoria y rendimiento práctico.


# Sección 1: ¿Por qué Quicksort después de Merge Sort? (5 minutos)

## Recap: lo que sabemos

En la clase anterior vimos **Merge Sort**:
- Paradigma: Divide & Conquer
- Complejidad: **O(n log n) siempre** (mejor, promedio y peor caso)
- Memoria extra: **O(n)** — necesita arreglo auxiliar

Entonces... ¿para qué aprender otro algoritmo O(n log n)?

> 🎙️ **[PAUSA PROFESOR]** *"¿Alguien tiene una hipótesis de por qué Quicksort podría ser mejor que Merge Sort en la práctica, si tienen la misma complejidad asintótica?"*

## La respuesta: las constantes importan

La notación O(n log n) **oculta factores constantes**:

| Algoritmo | Ops/elemento aprox. | Memoria extra | Cache-friendly |
|-----------|---------------------|---------------|----------------|
| Merge Sort | ~2n log n | O(n) | Moderado |
| Quicksort | ~1.4n log n (promedio) | O(log n) pila | **Sí** |

Quicksort es **in-place** y tiene mejor localidad de caché → en la práctica es ~2× más rápido que Merge Sort para datos en memoria RAM.

> 📌 Por eso `Arrays.sort()` en Java, `qsort()` en C, y variantes de Quicksort son la elección por defecto en la mayoría de las librerías estándar.

# Sección 2: La Idea Central — La Partición (10 minutos)

## El insight fundamental

Quicksort se basa en una observación simple:

> Si encontramos un elemento que ya está **en su posición final correcta**,  
> podemos ordenar el resto de forma independiente.

¿Cuándo un elemento está en su posición final?  
**Cuando todos los elementos menores están a su izquierda y todos los mayores a su derecha.**

A ese elemento lo llamamos **pivot**.

## ¿Qué es una partición?

Dado un arreglo y un pivot elegido:
```
ANTES:  [3, 6, 8, 10, 1, 2, 1]   pivot = 3

DESPUÉS: [1, 2, 1,  3,  6, 8, 10]
          ↑        ↑   ↑
       menores   pivot  mayores
```

La función `particionar(arr, inicio, fin)` debe:
1. Elegir un pivot
2. Reorganizar los elementos: menores antes, mayores después
3. **Devolver el índice final del pivot**

> 🎙️ **[PAUSA PROFESOR]** *"La clave es que el pivot queda en su posición definitiva. ¿Cuántas veces se moverá el pivot después de la partición?"*  
> *Respuesta esperada: nunca más, está en su lugar final.*

## La recursión de Quicksort

```
quicksort(arr, inicio, fin):
    si inicio < fin:
        p = particionar(arr, inicio, fin)  # p es el índice del pivot
        quicksort(arr, inicio, p - 1)      # ordenar la mitad izquierda
        quicksort(arr, p + 1, fin)         # ordenar la mitad derecha
```

La elegancia: **no hay paso "Combine"** — todo el trabajo es en la partición.

# Sección 3: La Partición de Dos Punteros (15 minutos)

## 3.1 La idea

Tomamos como **pivote el primer elemento**, `arr[lo]`, y hacemos avanzar dos punteros uno
contra el otro:

- **`i`** parte **al lado del pivote** (`lo + 1`) y **avanza** mientras los elementos sean
  **menores** que el pivote.
- **`j`** parte en **`hi`** y **retrocede** mientras los elementos sean **mayores** que el
  pivote.

Cuando ambos se detienen, cada uno está parado sobre un elemento que está **del lado
equivocado**: `arr[i]` es demasiado grande para estar a la izquierda y `arr[j]` demasiado
pequeño para estar a la derecha.

> 📌 **La observación central:** los dos punteros detenidos señalan exactamente una
> **inversión**. Un solo intercambio la corrige y ambos pueden seguir avanzando.

Se repite hasta que los punteros se cruzan. En ese momento `j` marca la frontera: todo lo
que quedó a su izquierda es ≤ pivote. El último paso es **intercambiar el pivote con
`arr[j]`**, con lo que el pivote aterriza en su **posición definitiva**.

```
arr = [3, 5, 7, 1, 2, 4, 6, 8]      pivote = arr[0] = 3

        i →                      ← j
   [3 | 5   7   1   2   4   6   8]
    ↑   i=1                    j=7
  pivote

Paso 1:  i se detiene en 1  (arr[1]=5, no es < 3)
         j retrocede 8, 6, 4 y se detiene en 4  (arr[4]=2, no es > 3)
         swap(arr[1], arr[4])  →  [3 | 2   7   1   5   4   6   8]

Paso 2:  i se detiene en 2  (arr[2]=7)
         j se detiene en 3  (arr[3]=1)
         swap(arr[2], arr[3])  →  [3 | 2   1   7   5   4   6   8]

Paso 3:  i=3, j=2  →  se cruzaron, termina el ciclo

Final:   swap(pivote, arr[j]) = swap(arr[0], arr[2])
         →  [1   2 | 3 | 7   5   4   6   8]
                     ↑
              pivote en su lugar definitivo (índice 2)
```

> 🎙️ **[PAUSA PROFESOR]** *"¿Por qué los punteros se detienen también cuando el elemento es
> **igual** al pivote, en vez de saltárselo?"*
> *Respuesta: si no se detuvieran, un arreglo con todas las claves iguales mandaría un puntero
> hasta el extremo y la partición quedaría completamente desbalanceada → O(n²). Detenerse en
> los iguales reparte los duplicados entre ambos lados.*


In [ ]:
def particionar(arr, lo, hi):
    """
    Partición de dos punteros con el pivote en el primer elemento.

    El pivote es arr[lo]. El puntero i avanza desde lo+1 mientras encuentre
    elementos menores que el pivote; el puntero j retrocede desde hi mientras
    encuentre elementos mayores. Cuando ambos se detienen han hallado una
    inversión y se corrige con un intercambio. Al cruzarse, el pivote se
    intercambia con arr[j] y queda en su posición definitiva.

    Parámetros:
        arr (list): arreglo a particionar (se modifica in-place)
        lo (int): índice inicial del segmento
        hi (int): índice final del segmento (inclusive)

    Retorna:
        int: índice donde quedó el pivote, ya en su posición definitiva

    Complejidad:
        Temporal: O(hi - lo) — cada puntero recorre el segmento una sola vez
        Espacial: O(1) — trabaja in-place, solo usa índices

    Ejemplo:
        >>> a = [3, 5, 7, 1, 2, 4, 6, 8]
        >>> particionar(a, 0, 7)
        2
        >>> a
        [1, 2, 3, 7, 5, 4, 6, 8]
    """
    pivote = arr[lo]
    i = lo + 1          # arranca al lado del pivote
    j = hi              # arranca en el extremo derecho

    while True:
        # i avanza mientras los elementos sean MENORES que el pivote
        while i <= j and arr[i] < pivote:
            i += 1
        # j retrocede mientras los elementos sean MAYORES que el pivote
        while i <= j and arr[j] > pivote:
            j -= 1

        if i >= j:      # se cruzaron: la partición está lista
            break

        # ambos punteros señalan una inversión → la corregimos
        arr[i], arr[j] = arr[j], arr[i]
        i += 1
        j -= 1

    # el pivote viaja a la frontera: su posición definitiva
    arr[lo], arr[j] = arr[j], arr[lo]
    return j


def quicksort(arr, lo=0, hi=None):
    """
    Quicksort in-place sobre la partición de dos punteros.

    Complejidad:
        Temporal: O(n log n) promedio, O(n²) peor caso
        Espacial: O(log n) de pila en promedio
    """
    if hi is None:
        hi = len(arr) - 1
    if lo < hi:
        p = particionar(arr, lo, hi)
        quicksort(arr, lo, p - 1)      # izquierda: todo ≤ pivote
        quicksort(arr, p + 1, hi)      # derecha:   todo ≥ pivote
    return arr


# Demostración
ejemplo = [3, 5, 7, 1, 2, 4, 6, 8]
print(f"Antes:   {ejemplo}")
print(f"Después: {quicksort(ejemplo.copy())}")

# Verificación sobre datos aleatorios
import random
for _ in range(200):
    prueba = [random.randint(0, 50) for _ in range(random.randint(0, 40))]
    assert quicksort(prueba.copy()) == sorted(prueba), f"❌ falló con {prueba}"
print("✅ 200 pruebas aleatorias correctas (incluye duplicados y arreglos vacíos)")


## 3.2 Trazando la partición paso a paso

> 🎙️ **[PAUSA PROFESOR]** *"Ejecutemos con [5, 3, 8, 1, 7] en la pizarra antes de correr la
> celda. ¿Dónde creen que va a quedar el 5?"*


In [ ]:
def particionar_verbose(arr, lo, hi):
    """Misma partición, mostrando cada movimiento de los punteros."""
    arr = arr.copy()
    pivote = arr[lo]
    i, j = lo + 1, hi
    print(f"  pivote = arr[{lo}] = {pivote}")
    print(f"  inicio: i={i}, j={j}   {arr}")

    while True:
        avanzados = []
        while i <= j and arr[i] < pivote:
            avanzados.append(f"arr[{i}]={arr[i]}<{pivote}")
            i += 1
        if avanzados:
            print(f"    i avanza sobre {', '.join(avanzados)} → i={i}")
        else:
            print(f"    i no avanza: arr[{i}]={arr[i]} no es menor que {pivote}")

        retrocedidos = []
        while i <= j and arr[j] > pivote:
            retrocedidos.append(f"arr[{j}]={arr[j]}>{pivote}")
            j -= 1
        if retrocedidos:
            print(f"    j retrocede sobre {', '.join(retrocedidos)} → j={j}")
        else:
            print(f"    j no retrocede: arr[{j}]={arr[j]} no es mayor que {pivote}")

        if i >= j:
            print(f"    i={i} y j={j} se cruzaron → fin del ciclo")
            break

        print(f"    inversión en i={i} y j={j} → swap({arr[i]}, {arr[j]})")
        arr[i], arr[j] = arr[j], arr[i]
        i += 1
        j -= 1
        print(f"       {arr}")

    arr[lo], arr[j] = arr[j], arr[lo]
    print(f"  pivote a su lugar: swap(arr[{lo}], arr[{j}]) → {arr}")
    print(f"  → el pivote {pivote} queda fijo en el índice {j}")
    return arr, j


ejemplo = [5, 3, 8, 1, 7]
print(f"Arreglo inicial: {ejemplo}\n")
resultado, idx = particionar_verbose(ejemplo, 0, len(ejemplo) - 1)
print(f"\n✅ Izquierda {resultado[:idx]} ≤ {resultado[idx]} ≤ {resultado[idx+1:]} derecha")


# Sección 4: Partición Tricotómica de Dijkstra (12 minutos)

## 4.1 El problema de las claves repetidas

La partición de dos punteros reparte bien los duplicados, pero **los sigue reprocesando**: si
el arreglo tiene solo tres valores distintos y un millón de elementos, Quicksort vuelve a
particionar una y otra vez segmentos donde ya está todo igual al pivote.

> 🎙️ **[PAUSA PROFESOR]** *"Si sé que arr[k] es igual al pivote, ¿tiene sentido volver a
> ordenar esa posición en las llamadas recursivas?"*
> *No: ya está en su lugar definitivo, igual que el pivote.*

## 4.2 La bandera holandesa

**Edsger Dijkstra** planteó el problema de la *bandera nacional holandesa*: ordenar un arreglo
de tres colores en una sola pasada. Aplicado a Quicksort, la partición produce **tres zonas**
en vez de dos:

```
        < pivote        == pivote        sin revisar        > pivote
   ┌──────────────┬─────────────────┬─────────────────┬──────────────┐
   │              │                 │                 │              │
   └──────────────┴─────────────────┴─────────────────┴──────────────┘
   lo           lt-1  lt          i-1  i             gt  gt+1        hi
```

Tres punteros mantienen el invariante:

| Rango | Contenido |
|-------|-----------|
| `arr[lo .. lt-1]` | **menores** que el pivote |
| `arr[lt .. i-1]` | **iguales** al pivote |
| `arr[i .. gt]` | **aún sin clasificar** |
| `arr[gt+1 .. hi]` | **mayores** que el pivote |

En cada paso se mira `arr[i]`:

- **menor** → `swap(arr[lt], arr[i])`, avanzan `lt` e `i`
- **mayor** → `swap(arr[i], arr[gt])`, retrocede `gt` (**`i` no avanza**: lo que llegó
  desde la derecha todavía no se ha revisado)
- **igual** → solo avanza `i`

Al terminar, **toda la zona central ya está en su posición definitiva** y la recursión se
aplica únicamente a `[lo, lt-1]` y `[gt+1, hi]`.

> 💡 **El resultado que importa:** con un número constante de claves distintas, Quicksort
> tricotómico pasa de O(n log n) a **O(n)**. Es la variante que usa `java.util.Arrays.sort`
> para primitivos desde hace años.


In [ ]:
def particionar_tres_vias(arr, lo, hi):
    """
    Partición tricotómica de Dijkstra (bandera nacional holandesa).

    Deja el arreglo con tres zonas: menores, iguales y mayores que el pivote.
    Toda la zona de iguales queda en su posición definitiva.

    Parámetros:
        arr (list): arreglo a particionar (se modifica in-place)
        lo (int): índice inicial del segmento
        hi (int): índice final del segmento (inclusive)

    Retorna:
        tuple[int, int]: (lt, gt) tal que arr[lt..gt] son todos iguales al pivote

    Complejidad:
        Temporal: O(hi - lo) — una sola pasada
        Espacial: O(1)

    Ejemplo:
        >>> a = [2, 1, 2, 3, 2, 0, 2]
        >>> particionar_tres_vias(a, 0, 6)
        (2, 4)
    """
    pivote = arr[lo]
    lt = lo         # frontera izquierda de la zona de iguales
    i = lo + 1      # elemento en revisión
    gt = hi         # frontera derecha de la zona sin revisar

    while i <= gt:
        if arr[i] < pivote:
            arr[lt], arr[i] = arr[i], arr[lt]
            lt += 1
            i += 1
        elif arr[i] > pivote:
            arr[i], arr[gt] = arr[gt], arr[i]
            gt -= 1          # ojo: i NO avanza, lo que llegó no se ha revisado
        else:
            i += 1

    return lt, gt


def quicksort_tres_vias(arr, lo=0, hi=None):
    """Quicksort con partición tricotómica. O(n) con pocas claves distintas."""
    if hi is None:
        hi = len(arr) - 1
    if lo < hi:
        lt, gt = particionar_tres_vias(arr, lo, hi)
        quicksort_tres_vias(arr, lo, lt - 1)     # solo los menores
        quicksort_tres_vias(arr, gt + 1, hi)     # solo los mayores
    return arr


# Verificación
for _ in range(200):
    prueba = [random.randint(0, 5) for _ in range(random.randint(0, 40))]
    assert quicksort_tres_vias(prueba.copy()) == sorted(prueba), f"❌ falló con {prueba}"
print("✅ 200 pruebas aleatorias correctas\n")

# ── Dónde se nota la diferencia: muchas claves repetidas ────────────────────
import sys
sys.setrecursionlimit(50000)

print(f"{'Claves distintas':>17} {'2 punteros':>13} {'tricotómico':>13} {'mejora':>9}")
print("-" * 56)
n = 20000
for distintas in (2, 5, 20, 1000, n):
    datos = [random.randint(0, distintas - 1) for _ in range(n)]

    a = datos.copy()
    t0 = time.perf_counter()
    quicksort(a)
    t_dos = time.perf_counter() - t0

    b = datos.copy()
    t0 = time.perf_counter()
    quicksort_tres_vias(b)
    t_tres = time.perf_counter() - t0

    assert a == b == sorted(datos)
    print(f"{distintas:>17,} {t_dos*1000:>11.1f}ms {t_tres*1000:>11.1f}ms {t_dos/t_tres:>8.1f}x")

print("\n💡 Con pocas claves distintas la ventaja es enorme.")
print("💡 Con todas las claves distintas el tricotómico PIERDE: mantiene una zona de")
print("   iguales que casi siempre tiene un solo elemento, y paga ese trabajo extra.")
print("   No hay almuerzo gratis: la variante se elige sabiendo cómo son los datos.")


# Sección 5: Estrategias de Pivot (8 minutos)

## ¿Por qué importa el pivot?

La elección del pivot determina el **balance de la partición**:

- **Pivot ideal:** divide en dos mitades iguales → árbol equilibrado → O(n log n)
- **Peor pivot:** siempre elige el mínimo o máximo → árbol degenerado → **O(n²)**

## Las 4 estrategias principales

In [ ]:
def pivote_primer_elemento(arr, lo, hi):
    """Pivote = arr[lo]. Es lo que hace nuestra partición. O(n²) si ya viene ordenado."""
    return arr[lo]

def pivote_aleatorio(arr, lo, hi):
    """Elige un pivote al azar y lo mueve a lo. Elimina el peor caso determinista."""
    idx = random.randint(lo, hi)
    arr[lo], arr[idx] = arr[idx], arr[lo]
    return arr[lo]

def pivote_mediana_de_tres(arr, lo, hi):
    """
    Pivote = mediana de arr[lo], arr[medio], arr[hi], movida a lo.
    Buen equilibrio en la práctica — es lo que usa introsort.
    """
    medio = (lo + hi) // 2
    if arr[lo] > arr[medio]:
        arr[lo], arr[medio] = arr[medio], arr[lo]
    if arr[lo] > arr[hi]:
        arr[lo], arr[hi] = arr[hi], arr[lo]
    if arr[medio] > arr[hi]:
        arr[medio], arr[hi] = arr[hi], arr[medio]
    arr[lo], arr[medio] = arr[medio], arr[lo]   # la mediana al frente
    return arr[lo]


def quicksort_con_pivote(arr, elegir_pivote, lo=0, hi=None, contador=None):
    """Quicksort parametrizado por la estrategia de pivote, contando comparaciones."""
    if hi is None:
        hi = len(arr) - 1
    if contador is None:
        contador = [0]
    if lo < hi:
        elegir_pivote(arr, lo, hi)      # deja el pivote elegido en arr[lo]
        contador[0] += hi - lo
        p = particionar(arr, lo, hi)
        quicksort_con_pivote(arr, elegir_pivote, lo, p - 1, contador)
        quicksort_con_pivote(arr, elegir_pivote, p + 1, hi, contador)
    return contador[0]


# El peor caso en acción: arreglo YA ORDENADO
n = 2000
ya_ordenado = list(range(n))
sys.setrecursionlimit(50000)

print(f"Arreglo ya ordenado, n = {n:,}\n")
print(f"{'Estrategia':<24} {'Comparaciones':>15} {'Referencia':>15}")
print("-" * 58)

for nombre, estrategia in [("Primer elemento", pivote_primer_elemento),
                           ("Aleatorio", pivote_aleatorio),
                           ("Mediana de tres", pivote_mediana_de_tres)]:
    arr = ya_ordenado.copy()
    try:
        comps = quicksort_con_pivote(arr, estrategia)
        assert arr == ya_ordenado
        print(f"{nombre:<24} {comps:>15,}")
    except RecursionError:
        print(f"{nombre:<24} {'RecursionError':>15}   ← recursión de profundidad n")

print(f"\n{'O(n log n) teórico':<24} {int(n * np.log2(n)):>15,}")
print(f"{'O(n²) teórico':<24} {n * n:>15,}")


## Resumen de estrategias de pivote

| Estrategia | Implementación | Peor caso | Promedio | Usado en |
|-----------|---------------|-----------|----------|---------|
| Primer elemento | Trivial | O(n²) con datos ordenados | O(n log n) | Ejercicios académicos |
| Aleatorio | `random.randint` | O(n²) improbable | O(n log n) | Cuando se desconfía del input |
| Mediana de 3 | 3 comparaciones | O(n²) muy raro | ~10% mejor que aleatorio | C++ `std::sort`, introsort |
| Mediana de medianas | O(n) | **O(n log n) garantizado** | O(n log n) | Análisis teórico |

> ⚠️ **El peor caso no es un arreglo raro:** un arreglo **ya ordenado** basta para degradar a
> O(n²) si el pivote es siempre el primer elemento. Y recibir datos ya ordenados es de lo más
> común en la práctica. Por eso ninguna implementación seria usa pivote fijo.

> 🎙️ **[PAUSA PROFESOR]** *"¿Por qué Python usa Timsort y no Quicksort para `sorted()`?
> Hint: piensen en estabilidad."*


# Sección 6: Análisis de Complejidad (8 minutos)

## 6.1 Los tres escenarios

### Peor caso — O(n²)
Ocurre cuando la partición es siempre completamente desbalanceada (pivot = mínimo o máximo):

$$T(n) = T(n-1) + T(0) + O(n) = T(n-1) + O(n)$$

Resolviendo: $T(n) = O(n^2)$

**Ejemplo:** Quicksort con pivot=último sobre arreglo **ya ordenado** o **inversamente ordenado**.

### Mejor caso — O(n log n)
Ocurre cuando el pivot siempre divide en dos mitades exactamente iguales:

$$T(n) = 2T(n/2) + O(n)$$

Por el Teorema Maestro (igual que Merge Sort): $T(n) = O(n \log n)$

### Caso promedio — O(n log n)
Para una permutación aleatoria, el pivot esperado divide en partes proporcionales.  
El análisis exacto da:

$$T(n) \approx 2n \ln n \approx 1.386 \cdot n \log_2 n$$

Esto es **~39% más que el caso ideal** pero sigue siendo O(n log n).

## 6.2 Complejidad espacial

| Caso | Espacio de pila |
|------|----------------|
| Mejor/Promedio | O(log n) |
| Peor | O(n) ← ¡posible stack overflow! |

> Con optimización **tail recursion** en la rama mayor, se garantiza O(log n) siempre.

In [ ]:
# Visualización empírica de las tres complejidades
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

tamaños = [50, 100, 200, 400, 800, 1600]
comps_aleatorio = []
comps_ordenado = []
comps_teorico = []

sys.setrecursionlimit(50000)

for n in tamaños:
    # Caso promedio: datos aleatorios
    datos = random.sample(range(n * 10), n)
    comps_aleatorio.append(quicksort_con_pivote(datos.copy(), pivote_primer_elemento))

    # Referencia teórica O(n log n)
    comps_teorico.append(int(n * np.log2(max(n, 2))))

    # Peor caso: datos ya ordenados con pivote fijo
    try:
        comps_ordenado.append(quicksort_con_pivote(list(range(n)), pivote_primer_elemento))
    except RecursionError:
        comps_ordenado.append(n * n // 2)      # estimación cuando la pila se agota

# Gráfico izquierdo: comparaciones absolutas
ax1 = axes[0]
ax1.plot(tamaños, comps_aleatorio, 'b-o', label='Datos aleatorios', linewidth=2)
ax1.plot(tamaños, comps_teorico, 'g--s', label='O(n log n) teórico', linewidth=2)
ax1.plot(tamaños, comps_ordenado, 'r-^', label='Datos ordenados (peor caso)', linewidth=2)
ax1.set_xlabel('n (tamaño)', fontsize=12)
ax1.set_ylabel('Comparaciones', fontsize=12)
ax1.set_title('Comparaciones vs n', fontsize=13)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Gráfico derecho: normalizado por n log n — si converge, es O(n log n)
ax2 = axes[1]
nlogn = [n * np.log2(max(n, 2)) for n in tamaños]
ax2.plot(tamaños, [c / t for c, t in zip(comps_aleatorio, nlogn)], 'b-o',
         label='Aleatorios / (n log n)', linewidth=2)
ax2.plot(tamaños, [c / t for c, t in zip(comps_ordenado, nlogn)], 'r-^',
         label='Ordenados / (n log n)', linewidth=2)
ax2.set_xlabel('n (tamaño)', fontsize=12)
ax2.set_ylabel('Comparaciones / (n log n)', fontsize=12)
ax2.set_title('Normalizado: constante ⇒ O(n log n)', fontsize=13)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("💡 La curva azul se aplana: con datos aleatorios el costo es Θ(n log n).")
print("💡 La curva roja crece sin parar: con datos ordenados y pivote fijo es Θ(n²).")


# Sección 7: Quicksort vs Merge Sort (5 minutos)

## Comparación teórica

| Criterio | Quicksort | Merge Sort |
|----------|-----------|------------|
| **Peor caso** | O(n²) | O(n log n) ✅ |
| **Caso promedio** | O(n log n) ✅ | O(n log n) ✅ |
| **Memoria extra** | O(log n) ✅ | O(n) |
| **Estable** | No ❌ | Sí ✅ |
| **In-place** | Sí ✅ | No (típicamente) |
| **Cache-friendly** | Sí ✅ | Moderado |
| **Constante práctica** | ~1.4n log n ✅ | ~2n log n |

In [ ]:
# Benchmark empírico: Quicksort vs Merge Sort
def merge_sort(arr):
    """Merge Sort del notebook anterior."""
    if len(arr) <= 1:
        return arr
    mitad = len(arr) // 2
    return merge(merge_sort(arr[:mitad]), merge_sort(arr[mitad:]))

def merge(izq, der):
    resultado = []
    i = j = 0
    while i < len(izq) and j < len(der):
        if izq[i] <= der[j]:
            resultado.append(izq[i]); i += 1
        else:
            resultado.append(der[j]); j += 1
    resultado.extend(izq[i:])
    resultado.extend(der[j:])
    return resultado


tamaños = [1000, 5000, 10000, 50000]
sys.setrecursionlimit(200000)

print(f"{'n':>7} {'Merge':>10} {'Quicksort':>11} {'sorted()':>10} {'Merge/Quick':>13}")
print("-" * 56)

for n in tamaños:
    datos = [random.randint(0, n * 10) for _ in range(n)]

    t0 = time.perf_counter()
    for _ in range(5):
        merge_sort(datos.copy())
    t_merge = (time.perf_counter() - t0) / 5 * 1000

    t0 = time.perf_counter()
    for _ in range(5):
        quicksort(datos.copy())          # partición de dos punteros
    t_quick = (time.perf_counter() - t0) / 5 * 1000

    t0 = time.perf_counter()
    for _ in range(5):
        sorted(datos)
    t_sorted = (time.perf_counter() - t0) / 5 * 1000

    print(f"{n:>7,} {t_merge:>8.2f}ms {t_quick:>9.2f}ms {t_sorted:>8.2f}ms {t_merge/t_quick:>12.2f}x")

print("\n💡 sorted() de Python usa Timsort (Merge Sort + Insertion Sort) — optimizado en C.")
print("💡 Nuestra implementación en Python es más lenta por el overhead del intérprete.")
print("💡 En C/C++ Quicksort es típicamente 2× más rápido que Merge Sort.")


# Sección 8: Resumen y Próxima Clase (2 minutos)

## ✅ Lo que aprendimos hoy

1. **Partición de dos punteros**: pivote en el primer elemento, `i` avanza desde la izquierda
   mientras encuentre menores, `j` retrocede desde la derecha mientras encuentre mayores, y
   cada vez que ambos se detienen se corrige una inversión con un intercambio.
2. **El pivote queda en su posición definitiva** tras el intercambio final: por eso no hay
   paso de combinación, a diferencia de Merge Sort.
3. **Estrategias de pivote**: aleatorio y mediana de tres evitan el peor caso O(n²), que se
   dispara con algo tan común como un arreglo ya ordenado.
4. **Complejidad**: O(n log n) promedio, O(n²) peor caso, O(log n) de pila.
5. **Partición tricotómica de Dijkstra**: tres zonas en vez de dos. Con pocas claves distintas
   baja de O(n log n) a **O(n)**.
6. **vs Merge Sort**: Quicksort gana en la práctica (in-place, buena localidad de caché);
   Merge Sort gana en garantías (peor caso y estabilidad).

## 🔑 Regla de decisión práctica

```
¿Necesito estabilidad?      →  Merge Sort (o Timsort)
¿RAM limitada?              →  Quicksort (in-place)
¿Muchas claves repetidas?   →  Quicksort tricotómico
¿Datos casi ordenados?      →  Timsort o Insertion Sort
¿Rendimiento puro?          →  Introsort (Quicksort + Heapsort + Insertion)
```

## 📚 Para el Laboratorio

Implementarás:

1. La partición de dos punteros desde cero
2. Quicksort completo sobre ella
3. Selección de pivote con mediana de tres
4. La partición tricotómica de Dijkstra
5. Comparación empírica entre las variantes

> 🎙️ **[PAUSA PROFESOR]** *"¿Preguntas antes del lab?"*
